# Modelo Causal v5 — IPTW (Inverse Probability of Treatment Weighting)

Baseado no artigo: *Causal Inference with DoWhy #4 — IPTW*  https://medium.com/data-science-explained/causal-inference-with-dowhy-4-inverse-probability-of-treatment-weighting-iptw-817de28e541f

## O que é IPTW?

IPTW atribui **pesos** a cada observação com base na probabilidade de receber o tratamento dado suas características (propensity score).

**Fórmula dos pesos (ATE):**
- Tratados (T=1): `w = 1 / e(X)`
- Controles (T=0): `w = 1 / (1 - e(X))`

onde `e(X) = P(T=1 | X)` é o propensity score.

**Intuição**: observações que receberam o tratamento de forma "surpreendente" (dado suas características) recebem peso maior — simulando um experimento aleatorizado.

## Análises neste notebook

Aplicamos IPTW nas mesmas análises dos notebooks anteriores para comparar resultados:

| Tratamento | Outcome | ATE anterior (Reg. Linear) |
|---|---|---|
| Despacho lento (>3d) | Entrega atrasada | +0.079 |
| Aprovação lenta (>24h) | Entrega atrasada | +0.016 |
| Despacho lento (>3d) | OSR | +0.005 |

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression

from app.config.settings import INTERIM_DATA_DIR
from dowhy import CausalModel

import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_parquet(os.path.join(INTERIM_DATA_DIR, "interim_dataset.parquet"))

# Converter colunas de data
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Variáveis operacionais
df["approval_time_hours"] = (
    df["order_approved_at"] - df["order_purchase_timestamp"]
).dt.total_seconds() / 3600

df["dispatch_time_days"] = (
    df["order_delivered_carrier_date"] - df["order_approved_at"]
).dt.days

df["delay_days"] = (
    df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]
).dt.days

# Tratamentos
df["aprovacao_lenta"] = (df["approval_time_hours"] > 24).astype(int)
df["despacho_lento"]  = (df["dispatch_time_days"] > 3).astype(int)

# Outcomes
df["entrega_atrasada"] = (df["delay_days"] > 0).astype(int)

print(df.shape)

In [ ]:
common_causes = [
    "total_price",
    "n_items",
    "avg_weight",
    "customer_state",
    "purchase_month",
    "purchase_weekday",
    "purchase_hour",
    "n_items_missing_info"
]

def preprocess(df, treatment, outcome, common_causes):
    cols = common_causes + [treatment, outcome]
    df_out = df[cols].dropna().copy()

    le = LabelEncoder()
    df_out["customer_state"] = le.fit_transform(df_out["customer_state"])

    continuous_cols = [c for c in common_causes if c != "customer_state"]
    scaler = StandardScaler()
    df_out[continuous_cols] = scaler.fit_transform(df_out[continuous_cols])

    return df_out

def build_graph(treatment, outcome, common_causes):
    edges = [f"    {treatment} -> {outcome};"]
    for c in common_causes:
        edges.append(f"    {c} -> {treatment};")
        edges.append(f"    {c} -> {outcome};")
    return "digraph {\n" + "\n".join(edges) + "\n}"

## Funções IPTW

Implementamos as funções auxiliares para:
1. Estimar propensity scores
2. Calcular pesos IPTW (com estabilização para evitar pesos extremos)
3. Verificar balanço das covariáveis (SMD — Standardized Mean Difference)
4. Estimar o ATE ponderado

In [ ]:
def compute_iptw_weights(df_model, treatment, common_causes, stabilized=True):
    """
    Calcula pesos IPTW para cada observação.
    
    - Tratados (T=1): w = 1 / e(X)  →  estabilizado: P(T=1) / e(X)
    - Controles (T=0): w = 1 / (1-e(X))  →  estabilizado: P(T=0) / (1-e(X))
    
    Pesos estabilizados reduzem a variância quando o propensity score
    é muito próximo de 0 ou 1.
    """
    X = df_model[common_causes].values
    T = df_model[treatment].values

    # Estima propensity score
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X, T)
    ps = lr.predict_proba(X)[:, 1]  # P(T=1 | X)

    # Clippa pesos extremos (evita instabilidade)
    ps = np.clip(ps, 0.01, 0.99)

    # Calcula pesos
    p_treated = T.mean()
    weights = np.where(
        T == 1,
        (p_treated / ps) if stabilized else (1 / ps),
        ((1 - p_treated) / (1 - ps)) if stabilized else (1 / (1 - ps))
    )

    return ps, weights


def compute_smd(df_model, treatment, common_causes, weights=None):
    """
    Calcula Standardized Mean Difference (SMD) para cada covariável.
    SMD < 0.1 indica bom balanço entre grupos.
    """
    smds = {}
    T = df_model[treatment].values

    for col in common_causes:
        x = df_model[col].values
        if weights is not None:
            mean1 = np.average(x[T == 1], weights=weights[T == 1])
            mean0 = np.average(x[T == 0], weights=weights[T == 0])
            var1  = np.average((x[T == 1] - mean1)**2, weights=weights[T == 1])
            var0  = np.average((x[T == 0] - mean0)**2, weights=weights[T == 0])
        else:
            mean1, var1 = x[T == 1].mean(), x[T == 1].var()
            mean0, var0 = x[T == 0].mean(), x[T == 0].var()

        pooled_std = np.sqrt((var1 + var0) / 2)
        smd = abs(mean1 - mean0) / pooled_std if pooled_std > 0 else 0
        smds[col] = smd

    return pd.Series(smds)


def ate_iptw(df_model, treatment, outcome, weights):
    """
    Estima o ATE usando pesos IPTW:
    ATE = média ponderada dos tratados - média ponderada dos controles
    """
    T = df_model[treatment].values
    Y = df_model[outcome].values

    ate = (
        np.average(Y[T == 1], weights=weights[T == 1]) -
        np.average(Y[T == 0], weights=weights[T == 0])
    )
    return ate


def bootstrap_ci(df_model, treatment, outcome, common_causes,
                 n_bootstrap=500, alpha=0.05, stabilized=True):
    """
    Calcula intervalo de confiança do ATE via bootstrap.
    """
    ates = []
    n = len(df_model)

    for _ in range(n_bootstrap):
        sample = df_model.sample(n=n, replace=True)
        _, w = compute_iptw_weights(sample, treatment, common_causes, stabilized)
        ates.append(ate_iptw(sample, treatment, outcome, w))

    lower = np.percentile(ates, 100 * alpha / 2)
    upper = np.percentile(ates, 100 * (1 - alpha / 2))
    return lower, upper, np.array(ates)

## Função principal de análise IPTW

In [ ]:
def run_iptw_analysis(df, treatment, outcome, common_causes, n_bootstrap=500):
    print(f"{'='*60}")
    print(f"IPTW: {treatment} → {outcome}")
    print(f"{'='*60}")

    df_model = preprocess(df, treatment, outcome, common_causes)

    # 1. Propensity scores e pesos
    ps, weights = compute_iptw_weights(df_model, treatment, common_causes)

    print(f"\nPropensity score — média: {ps.mean():.3f} | min: {ps.min():.3f} | max: {ps.max():.3f}")
    print(f"Pesos IPTW        — média: {weights.mean():.3f} | min: {weights.min():.3f} | max: {weights.max():.3f}")

    # 2. Balanço das covariáveis
    smd_antes  = compute_smd(df_model, treatment, common_causes)
    smd_depois = compute_smd(df_model, treatment, common_causes, weights=weights)

    df_smd = pd.DataFrame({
        "SMD antes": smd_antes.round(4),
        "SMD depois": smd_depois.round(4),
        "Balanceado?": smd_depois.apply(lambda x: "✅" if x < 0.1 else "❌")
    })
    print("\n--- Balanço das covariáveis (SMD < 0.1 = balanceado) ---")
    print(df_smd.to_string())

    # 3. ATE IPTW
    ate = ate_iptw(df_model, treatment, outcome, weights)
    print(f"\nATE (IPTW): {ate:.6f}")

    # 4. Intervalo de confiança via bootstrap
    print(f"Calculando IC via bootstrap ({n_bootstrap} amostras)...")
    lower, upper, boot_ates = bootstrap_ci(
        df_model, treatment, outcome, common_causes, n_bootstrap
    )
    print(f"IC 95%: [{lower:.6f}, {upper:.6f}]")

    # 5. Significância: IC não contém zero?
    significativo = "✅ Significativo" if (lower > 0 or upper < 0) else "❌ Não significativo (IC contém zero)"
    print(f"Resultado: {significativo}")

    # 6. Gráfico da distribuição bootstrap
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(boot_ates, bins=40, color="steelblue", alpha=0.7, edgecolor="white")
    ax.axvline(ate, color="red", linewidth=2, label=f"ATE = {ate:.4f}")
    ax.axvline(lower, color="orange", linestyle="--", linewidth=1.5, label=f"IC 95% inferior = {lower:.4f}")
    ax.axvline(upper, color="orange", linestyle="--", linewidth=1.5, label=f"IC 95% superior = {upper:.4f}")
    ax.axvline(0, color="black", linestyle=":", linewidth=1, label="Zero")
    ax.set_title(f"Distribuição Bootstrap do ATE\n{treatment} → {outcome}")
    ax.set_xlabel("ATE estimado")
    ax.set_ylabel("Frequência")
    ax.legend(fontsize=9)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.savefig(f"../../reports/figures/iptw_bootstrap_{treatment}_{outcome}.png", dpi=150)
    plt.show()

    return {"treatment": treatment, "outcome": outcome,
            "ate": ate, "ic_lower": lower, "ic_upper": upper}

## Análise 1 — Despacho lento → Entrega atrasada

Principal achado dos notebooks anteriores (ATE = +0.079 pela Regressão Linear).

In [ ]:
r1 = run_iptw_analysis(df, "despacho_lento", "entrega_atrasada", common_causes, n_bootstrap=500)

## Análise 2 — Aprovação lenta → Entrega atrasada

Efeito moderado encontrado anteriormente (ATE = +0.016 pela Regressão Linear).

In [ ]:
r2 = run_iptw_analysis(df, "aprovacao_lenta", "entrega_atrasada", common_causes, n_bootstrap=500)

## Análise 3 — Despacho lento → OSR

Resultado contra-intuitivo anterior (ATE = +0.005).

In [ ]:
r3 = run_iptw_analysis(df, "despacho_lento", "OSR", common_causes, n_bootstrap=500)

## Comparação final: Regressão Linear vs IPTW

In [ ]:
comparacao = pd.DataFrame([
    {"Tratamento": "despacho_lento",  "Outcome": "entrega_atrasada",
     "ATE Reg. Linear": 0.079,  "ATE IPTW": r1["ate"],
     "IC 95% inferior": r1["ic_lower"], "IC 95% superior": r1["ic_upper"]},
    {"Tratamento": "aprovacao_lenta", "Outcome": "entrega_atrasada",
     "ATE Reg. Linear": 0.016,  "ATE IPTW": r2["ate"],
     "IC 95% inferior": r2["ic_lower"], "IC 95% superior": r2["ic_upper"]},
    {"Tratamento": "despacho_lento",  "Outcome": "OSR",
     "ATE Reg. Linear": 0.005,  "ATE IPTW": r3["ate"],
     "IC 95% inferior": r3["ic_lower"], "IC 95% superior": r3["ic_upper"]},
])

print(comparacao.round(6).to_string(index=False))